## MULTITAN NMT Fine-tuning Pipeline (Google Colab-compatible)
### 1. Environment Setup
- Use a GPU running environment (and restart the kernel)
- Mount Google Drive to access files.
- Create a project directory.

### 2. Data Preparation
- Load local aligned bilingual datasets (TSV or CSV formats).
- Preprocess and filter the dataset based on:
  - Alignment quality
  - Segment length
  - Encoding issues

### 3. Model Fine-tuning Pipeline
- Fine-tune a **Seq2Seq model** (e.g., Facebook’s **NLLB 2B** or **Marian MT**).
- Uses the HuggingFace `transformers` library.
- Components involved:
  - `AutoModelForSeq2SeqLM`
  - `AutoTokenizer`
  - `Seq2SeqTrainer` and its training arguments
  - Data collators
  - Training configuration

### 4. Translation and machine evaluation
- Load a CSV or TSV file with test (source language) and reference (target language) segments
- Load both **pretrained** and **fine-tuned** models from saved paths.
- Translate the source segments using each model with:
  - Tokenization
  - Sequence generation (`generate()`)
  - Translation outputs (TSV)
- Evaluate translation quality using:
  - **BLEU** (via `sacrebleu`)
  - **chrF**
  - **COMET**




### Environment Setup
- Use a GPU runinng environment


In [ ]:
#create directory
!mkdir "MULTITAN_pipeline"

In [ ]:
#navigate to the directory
%cd "MULTITAN_pipeline"

### Import aligned segments (TSV or CSV)
#### Format
  - [x] aligned bilingual segments
  - [x] two columns with the headers `src` (source language) and `tgt` (target language)

In [ ]:
## Import local dataset for fine-tuning

## ‼️ You need to prepare a csv or tsv file of you aligned segments in two columns, one with header "src" for source language segments, the other with "tgt" for target language segments

from google.colab import files
files.upload()

In [ ]:
### Enter the uploaded file's name below (tsv or csv)
input_csv = "your_dataset.tsv"

# Filtering : alignment quality, length and encode
## Delete the lines which are too long (beyond threshold) et and have encoding error

In [ ]:
import pandas as pd
import tabulate
def segments_ratio_ok(src_text, tgt_text, min_segment_len, ratio_threshold, delimiter):
    src_segments = src_text.split(delimiter)
    tgt_segments = tgt_text.split(delimiter)

    if any(len(seg.strip()) < min_segment_len for seg in src_segments):
        return False
    if any(len(seg.strip()) < min_segment_len for seg in tgt_segments):
        return False

    if len(src_segments) != len(tgt_segments):
        return False

    for s, t in zip(src_segments, tgt_segments):
        s_clean = s.strip()
        t_clean = t.strip()
        if len(s_clean) == 0 or len(t_clean) == 0:
            return False
        seg_ratio = min(len(s_clean), len(t_clean)) / max(len(s_clean), len(t_clean))
        if seg_ratio <= ratio_threshold:
            return False
    return True

def is_encodable(text, encoding):
    """Checks if a string can be encoded with a specific encoding."""
    try:
        text.encode(encoding)
        return True
    except UnicodeEncodeError:
        return False

def filter_alignment(df, src="src", tgt="tgt", max_len=250, encoding="utf-8",
                     min_segment_len=2, ratio_threshold=0.6, delimiter="\n"):
    valid = (
        df[src].notna() &
        df[tgt].notna() &
        df[src].str.len().le(max_len) &
        df[tgt].str.len().le(max_len) &
        df[src].apply(lambda x: isinstance(x, str) and is_encodable(x, encoding)) &
        df[tgt].apply(lambda x: isinstance(x, str) and is_encodable(x, encoding))
    )
    df_valid = df[valid].copy()
    cond_ratio = df_valid.apply(
        lambda row: segments_ratio_ok(row[src], row[tgt], min_segment_len, ratio_threshold, delimiter),
        axis=1
    )
    return df_valid[cond_ratio].reset_index(drop=True)

if input_csv.endswith(".tsv"):
    df = pd.read_csv("MULTITAN_pipeline/"+input_csv, encoding="utf-8", sep="\t")
elif input.endswith(".csv"):
    df = pd.read_csv("MULTITAN_pipeline/"+input_csv, encoding="utf-8")
else:
    print("Format non traité")


df_clean = filter_alignment(df)

filtered = input_csv.replace(".csv", "-filtered.tsv")
print(f)

df_clean.to_csv(f'MULTITAN_pipeline/{filtered}', index=False, encoding="utf-8", sep="\t")


removed_df = df.drop(df_clean.index)

print("\nRandom sample of 5 deleted lines:")
if len(removed_df) >= 10:
    sample_removed = removed_df.sample(5)
else:
    sample_removed = removed_df.copy()

try:
    from tabulate import tabulate
    print(tabulate(sample_removed, headers='keys', tablefmt='psql', showindex=False))
except ImportError:
    print(sample_removed.to_string(index=False))

print("\nRandom sample of 5 well-aligned lines:")
if len(df_clean) >= 10:
    sample_clean = df_clean.sample(5)
else:
    sample_clean = df_clean.copy()


print(tabulate(sample_clean, headers='keys', tablefmt='psql', showindex=False))


## Fine-tuning with a `seq2seq` MT model
### Requirements

In [ ]:
!pip install torch transformers datasets evaluate sentencepiece accelerate sacremoses bitsandbytes sacrebleu unbabel-comet
!pip install -U trl
!pip install trl==0.7.4

### Fine-tune NLLB, large multilingual NMT model (Huggingface Acces token is required)


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import torch
import pandas as pd


MODEL_NAME = "facebook/nllb-200-distilled-600M" #model name
SRC_LANG = "YOUR_SRC_LANG" # source language, compulsory declaration
TGT_LANG = "YOUR_TGT_LANG" # target language, compulsory declaration
BATCH_SIZE = 2
MAX_LENGTH = 64

## Personal token required here. To obtain a huggingface token, see here : https://huggingface.co/docs/hub/security-tokens
access_token = "YOUR_HF_ACCESS_TOKEN"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=access_token)
tokenizer.tgt_lang = TGT_LANG


def load_corpus(csv_path=filtered):
    sep = "\t" if csv_path.endswith(".tsv") else ";" if csv_path.endswith(".csv") and ";" in open(csv_path).readline() else ","
    df = (
        pd.read_csv(csv_path, usecols=["src", "tgt"], sep=sep)
        .dropna()
        .assign(
            src=lambda x: x["src"].astype(str).str.strip(),
            tgt=lambda x: x["tgt"].astype(str).str.strip()
        )
        .query("src != '' and tgt != ''")
    )
    return Dataset.from_pandas(df.reset_index(drop=True))


def preprocess_function(examples, tokenizer):
    inputs = examples["src"]
    targets = examples["tgt"]
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def train():
    dataset = load_corpus()
    dataset = dataset.train_test_split(test_size=0.2)

    tokenized_datasets = dataset.map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=dataset["train"].column_names
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, token=access_token)
    model.gradient_checkpointing_enable()

    # Save pretrained model before fine-tuning
    model.save_pretrained("MULTITAN_pipeline/nllb-pretrained")
    tokenizer.save_pretrained("MULTITAN_pipeline/nllb-pretrained")
    print("Pretrained model saved in 'MULTITAN_pipeline/nllb-pretrained'")

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir="./nllb-checkpoints",
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        learning_rate=1e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        max_steps=1000,
        predict_with_generate=True,
        fp16=True,
        gradient_accumulation_steps=2,
        logging_steps=50,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        data_collator=data_collator,
    )

    trainer.train()

    # Save fine-tuned model after training
    trainer.save_model("MULTITAN_pipeline/nllb-ft")
    tokenizer.save_pretrained("MULTITAN_pipeline/nllb-ft")
    print("Fine-tuned model saved in 'MULTITAN_pipeline/nllb-ft'")

if __name__ == "__main__":
    train()

## Fine-tune MarianMT, lightweight model



In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import torch
import pandas as pd


MODEL_NAME = "Helsinki-NLP/opus-mt-fr-en" #unidirectional model
SRC_LANG = "fr" #not necessary for Marian, but I declare it anyway
TGT_LANG = "en" #not necessary for Marian, but I declare it anyway
BATCH_SIZE = 16
MAX_LENGTH = 128


## Personal token required here. To obtain a huggingface token, see here : https://huggingface.co/docs/hub/security-tokens
access_token = "YOUR_HF_ACCESS_TOKEN"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=access_token)
tokenizer.tgt_lang = TGT_LANG

def load_corpus(csv_path=filtered):
    sep = "\t" if csv_path.endswith(".tsv") else ";" if csv_path.endswith(".csv") and ";" in open(csv_path).readline() else ","
    df = (
        pd.read_csv(csv_path, usecols=["src", "tgt"], sep=sep)
        .dropna()
        .assign(
            src=lambda x: x["src"].astype(str).str.strip(),
            tgt=lambda x: x["tgt"].astype(str).str.strip()
        )
        .query("src != '' and tgt != ''")
    )
    return Dataset.from_pandas(df.reset_index(drop=True))

def preprocess_function(examples, tokenizer):
    inputs = examples["src"]
    targets = examples["tgt"]
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def train():
    dataset = load_corpus()
    dataset = dataset.train_test_split(test_size=0.2)

    tokenized_datasets = dataset.map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=dataset["train"].column_names
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, token=access_token)
    model.gradient_checkpointing_enable()

    # Save pretrained model before fine-tuning
    model.save_pretrained("MULTITAN_pipeline/marian-pretrained")
    tokenizer.save_pretrained("MULTITAN_pipeline/marian-pretrained")
    print("Pretrained model saved in 'MULTITAN_pipeline/marian-pretrained'")

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


    training_args = Seq2SeqTrainingArguments(
        output_dir="./marian_checkpoints",
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        learning_rate=1e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        max_steps=1000,
        predict_with_generate=True,
        fp16=True,
        gradient_accumulation_steps=2,
        logging_steps=50,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        data_collator=data_collator,
    )

    trainer.train()

    # Save fine-tuned model after training
    trainer.save_model("MULTITAN_pipeline/marian-ft")
    tokenizer.save_pretrained("MULTITAN_pipeline/marian-ft")
    print("Fine-tuned model saved in 'MULTITAN_pipeline/marian-ft'")

if __name__ == "__main__":
    train()

# Translation & machine evaluation




In [ ]:
## ‼️ upload test set (csv), two columns, first as src segments, second as tgt ref segments, no header
## ‼️ order of the columns in the file matters here !

from google.colab import files
files.upload()

In [ ]:
test_set = "your_test_set.tsv"

## Prediction by pretrained and fine-tuned models & evaluation

In [ ]:
import torch, numpy as np, pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

DEVICE                 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRETRAINED_MODEL_PATH  = "YOUR_PRETRAINED_MODEL_PATH" #argument of model.save_pretrained
FINETUNED_MODEL_PATH   = "YOUR_FINETUNED_MODEL_PATH" #argument of trainer.save_model
SEGMENTS_PATH          = test_set  # .csv or .tsv
SRC_LANG, TGT_LANG     = "YOUR_SRC_LANG", "YOUR_TGT_LANG"



def load_segments(path: str, header: bool = False):
    """CSV or TSV, 2 columns : [source, reference]."""
    sep = "\t" if path.endswith(".tsv") else ";" if ";" in open(path).readline() else ","
    df  = pd.read_csv(path, sep=sep, engine="python", header=0 if header else None)
    return df.iloc[:, 0].tolist(), df.iloc[:, 1].tolist()

def load_model(path: str, src: str, tgt: str):
    tok = AutoTokenizer.from_pretrained(path)
    tok.src_lang, tok.tgt_lang = src, tgt
    mdl = AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE)
    return mdl, tok

def translate(texts, model, tokenizer, tgt, max_len=128):
    bos_id = tokenizer.convert_tokens_to_ids(f"<<{tgt}>>")
    outputs = []
    for t in tqdm(texts, desc="Translating"):
        inputs = tokenizer(t, return_tensors="pt", truncation=True).to(DEVICE)
        ids    = model.generate(**inputs, forced_bos_token_id=bos_id, max_length=max_len)
        outputs.append(tokenizer.decode(ids[0], skip_special_tokens=True))
    return outputs

def evaluate_all(preds, refs, srcs):
    bleu  = evaluate.load("sacrebleu")
    chrf  = evaluate.load("chrf")
    comet = evaluate.load("comet")
    return {
        "BLEU"  : bleu.compute(predictions=preds, references=[[r] for r in refs])["score"],
        "chrF"  : chrf.compute(predictions=preds, references=[[r] for r in refs])["score"],
        "COMET" : float(np.mean(comet.compute(predictions=preds, references=refs, sources=srcs)["scores"]))
    }


def main():

    texts, refs = load_segments(SEGMENTS_PATH, header=HAS_HEADER)

    base_model, base_tok = load_model(PRETRAINED_MODEL_PATH, SRC_LANG, TGT_LANG)
    ft_model,   ft_tok   = load_model(FINETUNED_MODEL_PATH,  SRC_LANG, TGT_LANG)

    base_trans = translate(texts, base_model, base_tok, TGT_LANG)
    ft_trans   = translate(texts, ft_model,   ft_tok,   TGT_LANG)

    df = pd.DataFrame({
    "source"                : texts,
    "translation_pretrained": base_trans,
    "translation_finetuned" : ft_trans
    })

    OUTPUT_PATH = "YOUR_OUTPUT_PATH"
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", sep="\t")
    print(f"File {OUTPUT_PATH} created.")

    base_scores = evaluate_all(base_trans, refs, texts)
    ft_scores   = evaluate_all(ft_trans,   refs, texts)

    print("\n=== Comparisons ===")
    for m in base_scores:
        print(f"{m:6s} |  Pre-trained : {base_scores[m]:6.2f}   Fine-tuned : {ft_scores[m]:6.2f}")

if __name__ == "__main__":
    main()